# ProjectI results

Volatility dynamics research and systematic vol strategies. Reads `results/*.json` and `data/derived/*.parquet` written by `python -m voldyn all`.

In [ ]:
import json, os, sys
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
sys.path.insert(0, os.path.abspath('..'))
from voldyn import data
R = os.path.abspath('../results'); D = os.path.abspath('../data/derived')
load = lambda n: json.load(open(os.path.join(R, n)))
bd, fc, pr, ev, vx, st = (load(n) for n in ('build.json', 'forecast.json', 'premia.json', 'events.json', 'vix.json', 'strategies.json'))

## 1. The variance risk premium by asset

In [ ]:
rows = [{'index': k, **{kk: v[kk] for kk in ('label', 'asset_class', 'n', 'gap_vol_mean_pts', 'share_positive')}, 'ci_lo': v['gap_vol_ci_pts'][0], 'ci_hi': v['gap_vol_ci_pts'][1], 'slope': v['rv_on_iv']['slope']} for k, v in pr['cross_asset'].items()]
pd.DataFrame(rows).sort_values('gap_vol_mean_pts', ascending=False)

In [ ]:
for k in ('VIX', 'DVOL_BTC', 'OVX'):
    by = pr['cross_asset'][k]['by_year']; plt.plot([int(y) for y in by], [v['gap_vol_pts'] for v in by.values()], 'o-', label=pr['cross_asset'][k]['label'])
plt.axhline(0, color='k', lw=0.6); plt.legend(); plt.ylabel('implied minus realised, vol points'); plt.show()

## 2. Forecasting

In [ ]:
pd.DataFrame({k: {m: v['models'][m]['qlike'] for m in v['models']} for k, v in fc['assets'].items()}).T

## 3. Events

In [ ]:
print(json.dumps({k: v for k, v in ev['fomc'].items() if k != 'paths'}, indent=1))
if 'earnings' in ev: display(pd.DataFrame(ev['earnings']['by_year']).T)

In [ ]:
e = pd.read_parquet(os.path.join(D, 'events_earnings.parquet')) if os.path.exists(os.path.join(D, 'events_earnings.parquet')) else None
if e is not None:
    plt.scatter(e['stripped_move'] * 100, e['abs_real'] * 100, s=5, alpha=0.4); plt.plot([0, 20], [0, 20], 'k:'); plt.xlim(0, 20); plt.ylim(0, 20); plt.xlabel('implied move %'); plt.ylabel('realised |move| %'); plt.show()

## 4. The VIX complex

In [ ]:
curve = pd.read_parquet(os.path.join(D, 'vx_curve.parquet')).set_index('date'); led = pd.read_parquet(os.path.join(D, 'vx_roll_ledger.parquet')).set_index('date')
led[['cum_gross', 'cum_net']].plot(title='short front VX, one contract, index points'); plt.show()
print(json.dumps(vx['roll_strategy'], indent=1)[:1500])

## 5. Strategies

In [ ]:
def tab(d):
    return {k: v for k, v in d.items() if k in ('n_trades', 'years', 'hit_rate', 'net_per_trade_pct_spot', 'gross_per_trade_pct_spot', 'cost_per_trade_pct_spot', 'return_on_margin_pct', 'max_drawdown_pct_margin', 'daily_sharpe', 'worst_trade_pct_spot')}
rows = {}
if 'spy_straddle' in st:
    rows['spy_straddle'] = tab(st['spy_straddle']['unconditional'])
for k in ('spy_calendar', 'btc_straddle', 'btc_straddle_costless', 'eth_straddle'):
    if k in st and st[k].get('n_trades'): rows[k] = tab(st[k])
pd.DataFrame(rows).T

In [ ]:
for name in ('spy_straddle', 'btc_straddle'):
    p = os.path.join(D, f'strategy_{name}_daily.parquet')
    if os.path.exists(p):
        d = pd.read_parquet(p).groupby('date')[['net', 'opt_pnl', 'hedge_pnl', 'cost']].sum()
        plt.plot(d.index, d['net'].cumsum(), label=f'{name} net'); plt.plot(d.index, (d['opt_pnl'] + d['hedge_pnl']).cumsum(), ':', label=f'{name} gross')
plt.legend(); plt.show()